In [8]:
import torch
from torch import nn

MAX_OTHERS = 128
MAX_SPOTS = 128
D_MODEL = 64

NUM_ITEMS = 2 #needs to sync with Rust

# Memory + Probs

In [4]:
from torch._higher_order_ops import scan


def sample_stochastic_gate(logits):
    """
    Continuous stochastic gate in (0, 1).
    """
    u = torch.rand_like(logits).clamp_(1e-6, 1.0 - 1e-6)
    noise = torch.log(u) - torch.log1p(-u)
    return torch.sigmoid(logits + noise)


class AproxMemory(nn.Module):
    def __init__(
        self,
        input_size,
        hidden_size,
        output_size,
        K,
    ):
        super().__init__()

        self.hidden_size = hidden_size
        self.output_size = output_size
        self.K = K

        # i, f, candidate
        self.x_proj = nn.Linear(input_size, 3 * hidden_size)
        self.h_proj = nn.Linear(hidden_size, 3 * hidden_size, bias=False)

        self.out_proj = nn.Linear(hidden_size, output_size)

    def forward(self, x, state=None):
        """
        x:
            (T, B, K, D)

        state:
            optional recurrent state:
            (B, K, H)

        returns:
            output:
                (T, B, K, O)

            state:
                final recurrent state
                (B, K, H)

            hidden:
                recurrent state at every timestep
                (T, B, K, H)

            activity:
                mean(i + f) per merchant
                (T, B, K)
        """
        T, B, K, _ = x.shape

        if K != self.K:
            raise ValueError(f"Expected K={self.K}, got K={K}")

        if state is None:
            state = x.new_zeros(B, K, self.hidden_size)

        def step(c, x_t):
            """
            c:
                (B, K, H)

            x_t:
                (B, K, D)
            """

            h = torch.tanh(c)

            # Linear naturally treats B and K as batch dimensions.
            gates = self.x_proj(x_t) + self.h_proj(h)

            i_logits, f_logits, g_logits = gates.chunk(3, dim=-1)

            i = sample_stochastic_gate(i_logits)
            f = sample_stochastic_gate(f_logits)
            g = torch.tanh(g_logits)

            c_new = f * c + i * g
            h_new = torch.tanh(c_new)

            y = self.out_proj(h_new)

            # (B, K)
            activity = (i + f).mean(dim=-1)

            return c_new, (y, h_new, activity)

        state, (output, hidden, activity) = scan(
            step,
            state,
            x,
        )

        return output, state, hidden, activity

In [5]:
AproxMemory(10,10,10,MAX_OTHERS)

AproxMemory(
  (x_proj): Linear(in_features=10, out_features=30, bias=True)
  (h_proj): Linear(in_features=10, out_features=30, bias=False)
  (out_proj): Linear(in_features=10, out_features=10, bias=True)
)

## PROBS

In [6]:
# action_layer.py

from __future__ import annotations

from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor


EPS = 1e-8
XI_EPS = 1e-5


# ============================================================
# Parameter / action structures
# ============================================================

@dataclass
class BarterSideParams:
    # [..., NUM_ITEMS + 1]
    # K = number of distinct item kinds.
    k_logits: Tensor

    # [..., NUM_ITEMS]
    #
    # IMPORTANT:
    # This serves TWO purposes:
    #
    #   1. logits for selecting which items participate
    #   2. GPD location for the quantity of that item
    #
    # So a high value means:
    #   "select this item more often"
    #   "and tend toward larger amounts"
    item_logits: Tensor

    # [..., NUM_ITEMS]
    amount_scale: Tensor

    # [..., NUM_ITEMS]
    amount_shape: Tensor


@dataclass
class BarterParams:
    buy: BarterSideParams
    sell: BarterSideParams


@dataclass
class TradeCountParams:
    # Probability that count > 0.
    # [...]
    execute_logit: Tensor

    # GPD for count, conditional on count > 0.
    # [...]
    location: Tensor
    scale: Tensor
    shape: Tensor


@dataclass
class ActionParams:
    # [..., 3]
    mode_logits: Tensor

    trade_count: TradeCountParams

    barter: BarterParams

    # [..., D_MODEL]
    merchant_query: Tensor

    # [..., D_MODEL]
    city_query: Tensor


@dataclass
class BarterSideAction:
    # [..., NUM_ITEMS]
    #
    # 0 = this item is absent from this side.
    quantities: Tensor

    # [...]
    K: Tensor

    # [..., NUM_ITEMS]
    #
    # Weighted sampling without replacement produces an ORDER.
    # The game does not care about this order, but PPO/etc. needs it
    # to reproduce the exact probability of the sampled action.
    item_order: Tensor


@dataclass
class BarterAction:
    buy: BarterSideAction
    sell: BarterSideAction


# ============================================================
# Neural action head
# ============================================================

class ActionLayer(nn.Module):
    def __init__(
        self,
        in_features: int,
        num_items: int = NUM_ITEMS,
        d_model: int = D_MODEL,
    ):
        super().__init__()

        self.num_items = num_items
        self.d_model = d_model

        # One barter side contains:
        #
        #   K logits                    num_items + 1
        #   item logits / GPD location  num_items
        #   GPD scale                   num_items
        #   GPD shape                   num_items
        #
        barter_side_size = (
            (num_items + 1)
            + num_items
            + num_items
            + num_items
        )

        out_features = (
            3                       # mode
            + 4                     # existing trade count
            + 2 * barter_side_size  # buy + sell
            + d_model               # merchant query
            + d_model               # city query
        )

        self.f = nn.Linear(in_features, out_features)

    def forward(self, x: Tensor) -> ActionParams:
        x = self.f(x)

        n = self.num_items
        d = self.d_model

        i = 0

        # ----------------------------------------------------
        # Mode
        # ----------------------------------------------------

        mode_logits = x[..., i:i + 3]
        i += 3

        # ----------------------------------------------------
        # Existing trade execution count
        # ----------------------------------------------------

        execute_logit = x[..., i]
        i += 1

        trade_location = x[..., i]
        i += 1

        trade_scale = F.softplus(x[..., i]) + EPS
        i += 1

        trade_shape = gpd_shape_from_raw(x[..., i])
        i += 1

        trade_count = TradeCountParams(
            execute_logit=execute_logit,
            location=trade_location,
            scale=trade_scale,
            shape=trade_shape,
        )

        # ----------------------------------------------------
        # Buy barter side
        # ----------------------------------------------------

        buy, i = self._read_barter_side(x, i)

        # ----------------------------------------------------
        # Sell barter side
        # ----------------------------------------------------

        sell, i = self._read_barter_side(x, i)

        # ----------------------------------------------------
        # Merchant / city queries
        # ----------------------------------------------------

        merchant_query = x[..., i:i + d]
        i += d

        city_query = x[..., i:i + d]
        i += d

        assert i == x.shape[-1]

        return ActionParams(
            mode_logits=mode_logits,
            trade_count=trade_count,
            barter=BarterParams(
                buy=buy,
                sell=sell,
            ),
            merchant_query=merchant_query,
            city_query=city_query,
        )

    def _read_barter_side(
        self,
        x: Tensor,
        i: int,
    ) -> tuple[BarterSideParams, int]:

        n = self.num_items

        k_logits = x[..., i:i + n + 1]
        i += n + 1

        # Shared between selection and quantity location.
        item_logits = x[..., i:i + n]
        i += n

        amount_scale = F.softplus(
            x[..., i:i + n]
        ) + EPS
        i += n

        amount_shape = gpd_shape_from_raw(
            x[..., i:i + n]
        )
        i += n

        return BarterSideParams(
            k_logits=k_logits,
            item_logits=item_logits,
            amount_scale=amount_scale,
            amount_shape=amount_shape,
        ), i


# ============================================================
# GPD parameterization
# ============================================================

def gpd_shape_from_raw(
    raw: Tensor,
    max_shape: float = 0.49,
) -> Tensor:
    """
    Positive Pareto-like tail.

    xi > 0:
        heavy Pareto tail

    xi < 1:
        finite mean

    xi < 0.5:
        finite variance

    0 < xi < 0.49 therefore gives us a genuinely Pareto-like
    distribution while keeping mean and variance finite.

    If you later WANT infinite-variance tails, increase max_shape.
    """

    return max_shape * torch.sigmoid(raw)


# ============================================================
# Continuous generalized Pareto distribution
# ============================================================

def gpd_log_survival(
    x: Tensor,
    location: Tensor,
    scale: Tensor,
    shape: Tensor,
) -> Tensor:
    """
    log P(X > x)

    GPD:

        S(x) =
            1                                       x < location

            (1 + xi*(x-location)/scale)^(-1/xi)    xi != 0

            exp(-(x-location)/scale)                xi == 0
    """

    y = x - location

    below_support = y < 0
    near_zero = shape.abs() < XI_EPS

    exponential = -y / scale

    z = 1.0 + shape * y / scale

    general = (
        -torch.log(z.clamp_min(EPS))
        / shape
    )

    log_survival = torch.where(
        near_zero,
        exponential,
        general,
    )

    # Below the lower support:
    #
    # P(X > x) = 1
    # log(1) = 0
    log_survival = torch.where(
        below_support,
        torch.zeros_like(log_survival),
        log_survival,
    )

    # General implementation still handles xi < 0,
    # even though our current network emits xi > 0.
    above_upper_support = (
        (shape < 0)
        & (z <= 0)
        & ~below_support
    )

    log_survival = torch.where(
        above_upper_support,
        torch.full_like(
            log_survival,
            -torch.inf,
        ),
        log_survival,
    )

    return log_survival


def gpd_survival(
    x: Tensor,
    location: Tensor,
    scale: Tensor,
    shape: Tensor,
) -> Tensor:
    return torch.exp(
        gpd_log_survival(
            x,
            location,
            scale,
            shape,
        )
    )


def gpd_cdf(
    x: Tensor,
    location: Tensor,
    scale: Tensor,
    shape: Tensor,
) -> Tensor:
    """
    P(X <= x)
    """

    return 1.0 - gpd_survival(
        x,
        location,
        scale,
        shape,
    )


def sample_gpd(
    location: Tensor,
    scale: Tensor,
    shape: Tensor,
) -> Tensor:
    """
    Inverse-CDF sampling.
    """

    u = torch.rand_like(location).clamp(
        EPS,
        1.0 - EPS,
    )

    near_zero = shape.abs() < XI_EPS

    # xi -> 0 is exponential.
    exponential = (
        location
        - scale * torch.log1p(-u)
    )

    general = (
        location
        + scale / shape
        * (
            (1.0 - u).pow(-shape)
            - 1.0
        )
    )

    return torch.where(
        near_zero,
        exponential,
        general,
    )


# ============================================================
# Numerically stable probability helpers
# ============================================================

def log1mexp(x: Tensor) -> Tensor:
    """
    log(1 - exp(x)), for x <= 0.
    """

    cutoff = -0.6931471805599453  # -log(2)

    return torch.where(
        x < cutoff,
        torch.log1p(-torch.exp(x)),
        torch.log(-torch.expm1(x)),
    )


def log_sub_exp(
    log_a: Tensor,
    log_b: Tensor,
) -> Tensor:
    """
    log(exp(log_a) - exp(log_b))

    Requires a >= b.

    Used for:

        S(lower) - S(upper)
    """

    valid = log_a > log_b

    safe_delta = torch.where(
        valid,
        log_b - log_a,
        torch.full_like(log_a, -1.0),
    )

    value = (
        log_a
        + log1mexp(safe_delta)
    )

    return torch.where(
        valid,
        value,
        torch.full_like(
            value,
            -torch.inf,
        ),
    )


def gpd_log_cdf(
    x: Tensor,
    location: Tensor,
    scale: Tensor,
    shape: Tensor,
) -> Tensor:
    """
    log P(X <= x)
    """

    log_s = gpd_log_survival(
        x,
        location,
        scale,
        shape,
    )

    # CDF = 1 - survival
    return log1mexp(log_s)


# ============================================================
# Positive integer censored GPD
#
# Used for quantities of SELECTED barter items.
#
# Values:
#
#     1 ... maximum
#
# We sample continuous X, then:
#
#     q = floor(X)
#     q = clamp(q, 1, maximum)
#
# Therefore:
#
# q = 1:
#     X < 2
#
# 1 < q < maximum:
#     q <= X < q+1
#
# q = maximum:
#     X >= maximum
#
# If maximum == 1, q=1 with probability 1.
# ============================================================

def sample_positive_gpd(
    location: Tensor,
    scale: Tensor,
    shape: Tensor,
    maximum: Tensor,
) -> Tensor:

    x = sample_gpd(
        location,
        scale,
        shape,
    )

    q = torch.floor(x).long()

    q = q.clamp_min(1)

    return torch.minimum(
        q,
        maximum.long(),
    )


def positive_gpd_log_prob(
    q: Tensor,
    location: Tensor,
    scale: Tensor,
    shape: Tensor,
    maximum: Tensor,
) -> Tensor:

    qf = q.to(scale.dtype)
    maximum = maximum.long()

    # --------------------------------------------
    # q == 1:
    #
    # Everything below 2 maps to 1.
    # --------------------------------------------

    first_logp = gpd_log_cdf(
        torch.full_like(qf, 2.0),
        location,
        scale,
        shape,
    )

    # --------------------------------------------
    # 1 < q < maximum:
    #
    # q <= X < q+1
    # --------------------------------------------

    log_s_lower = gpd_log_survival(
        qf,
        location,
        scale,
        shape,
    )

    log_s_upper = gpd_log_survival(
        qf + 1.0,
        location,
        scale,
        shape,
    )

    middle_logp = log_sub_exp(
        log_s_lower,
        log_s_upper,
    )

    # --------------------------------------------
    # q == maximum:
    #
    # Everything >= maximum maps to maximum.
    # --------------------------------------------

    max_logp = log_s_lower

    result = torch.where(
        q == 1,
        first_logp,
        middle_logp,
    )

    result = torch.where(
        q == maximum,
        max_logp,
        result,
    )

    # Special case:
    #
    # maximum == 1 means every possible sample becomes 1.
    result = torch.where(
        maximum == 1,
        torch.zeros_like(result),
        result,
    )

    valid = (
        (maximum >= 1)
        & (q >= 1)
        & (q <= maximum)
    )

    return torch.where(
        valid,
        result,
        torch.full_like(
            result,
            -torch.inf,
        ),
    )


# ============================================================
# Existing trade count
#
# Hurdle distribution:
#
#     execute? ~ Bernoulli
#
#     if no:
#         count = 0
#
#     if yes:
#         count ~ positive censored GPD
#
# This means the GPD itself NEVER needs to represent zero.
# ============================================================

def sample_trade_count(
    params: TradeCountParams,
    maximum: Tensor,
) -> Tensor:

    maximum = maximum.long()

    can_execute = maximum > 0

    execute = (
        torch.rand_like(params.execute_logit)
        < torch.sigmoid(params.execute_logit)
    )

    positive_count = sample_positive_gpd(
        params.location,
        params.scale,
        params.shape,
        maximum.clamp_min(1),
    )

    return torch.where(
        can_execute & execute,
        positive_count,
        torch.zeros_like(positive_count),
    )


def trade_count_log_prob(
    params: TradeCountParams,
    count: Tensor,
    maximum: Tensor,
) -> Tensor:

    maximum = maximum.long()

    # If execution is impossible, count=0 is forced.
    impossible = maximum == 0

    log_no = F.logsigmoid(
        -params.execute_logit
    )

    log_yes = F.logsigmoid(
        params.execute_logit
    )

    positive_logp = positive_gpd_log_prob(
        count.clamp_min(1),
        params.location,
        params.scale,
        params.shape,
        maximum.clamp_min(1),
    )

    result = torch.where(
        count == 0,
        log_no,
        log_yes + positive_logp,
    )

    forced = torch.where(
        count == 0,
        torch.zeros_like(result),
        torch.full_like(
            result,
            -torch.inf,
        ),
    )

    result = torch.where(
        impossible,
        forced,
        result,
    )

    valid = (
        (count >= 0)
        & (count <= maximum)
    )

    return torch.where(
        valid,
        result,
        torch.full_like(
            result,
            -torch.inf,
        ),
    )


# ============================================================
# Weighted item sampling WITHOUT replacement
#
# Gumbel top-k gives the same ordered distribution as
# sequential weighted categorical sampling without replacement
# (Plackett-Luce).
# ============================================================

def sample_item_order(
    item_logits: Tensor,
    available: Tensor,
) -> Tensor:

    logits = item_logits.masked_fill(
        ~available,
        -torch.inf,
    )

    u = torch.rand_like(logits).clamp(
        EPS,
        1.0 - EPS,
    )

    gumbel = (
        -torch.log(
            -torch.log(u)
        )
    )

    return torch.argsort(
        logits + gumbel,
        dim=-1,
        descending=True,
    )


def item_order_log_prob(
    item_logits: Tensor,
    item_order: Tensor,
    available: Tensor,
    K: Tensor,
) -> Tensor:
    """
    Exact probability of the ordered first-K selection.

    We retain item_order in the sampled action because the game
    only cares about the resulting set, but PPO/etc. needs the
    sampling path to compute exactly the same policy probability.
    """

    logits = item_logits.masked_fill(
        ~available,
        -torch.inf,
    )

    work = logits.clone()

    result = torch.zeros(
        logits.shape[:-1],
        dtype=logits.dtype,
        device=logits.device,
    )

    num_items = logits.shape[-1]

    for step in range(num_items):
        active = step < K

        index = item_order[..., step]

        chosen_logit = work.gather(
            -1,
            index.unsqueeze(-1),
        ).squeeze(-1)

        denominator = torch.logsumexp(
            work,
            dim=-1,
        )

        step_logp = (
            chosen_logit
            - denominator
        )

        result = result + torch.where(
            active,
            step_logp,
            torch.zeros_like(step_logp),
        )

        work.scatter_(
            -1,
            index.unsqueeze(-1),
            -torch.inf,
        )

    return result


# ============================================================
# K distribution
# ============================================================

def masked_k_logits(
    k_logits: Tensor,
    available_count: Tensor,
) -> Tensor:
    """
    k_logits[..., k] corresponds directly to K == k.

    Any K larger than the number of available item kinds is
    impossible.
    """

    k_values = torch.arange(
        k_logits.shape[-1],
        device=k_logits.device,
    )

    valid = (
        k_values
        <= available_count.unsqueeze(-1)
    )

    return k_logits.masked_fill(
        ~valid,
        -torch.inf,
    )


# ============================================================
# One barter side
# ============================================================

def sample_barter_side(
    params: BarterSideParams,
    maximum: Tensor,
) -> BarterSideAction:
    """
    maximum:
        [..., NUM_ITEMS]

    For the SELL side this is normally our inventory.

    For the BUY side this can represent whatever actually limits
    acquisition:
        seller inventory
        order availability
        cargo capacity
        etc.
    """

    maximum = maximum.long()

    available = maximum > 0

    available_count = available.sum(
        dim=-1
    )

    # --------------------------------------------------------
    # Sample K
    # --------------------------------------------------------

    k_logits = masked_k_logits(
        params.k_logits,
        available_count,
    )

    K = torch.distributions.Categorical(
        logits=k_logits,
    ).sample()

    # --------------------------------------------------------
    # Select K distinct items
    # --------------------------------------------------------

    item_order = sample_item_order(
        params.item_logits,
        available,
    )

    # Turn ordering into rank-per-item.
    ranks = torch.empty_like(
        item_order
    )

    positions = torch.arange(
        item_order.shape[-1],
        device=item_order.device,
    ).expand_as(item_order)

    ranks.scatter_(
        -1,
        item_order,
        positions,
    )

    selected = (
        ranks
        < K.unsqueeze(-1)
    )

    # --------------------------------------------------------
    # Sample quantity for every item.
    #
    # item_logits is ALSO the GPD location.
    # --------------------------------------------------------

    amounts = sample_positive_gpd(
        location=params.item_logits,
        scale=params.amount_scale,
        shape=params.amount_shape,
        maximum=maximum.clamp_min(1),
    )

    quantities = torch.where(
        selected,
        amounts,
        torch.zeros_like(amounts),
    )

    return BarterSideAction(
        quantities=quantities,
        K=K,
        item_order=item_order,
    )


def barter_side_log_prob(
    params: BarterSideParams,
    action: BarterSideAction,
    maximum: Tensor,
) -> Tensor:

    maximum = maximum.long()

    available = maximum > 0

    available_count = available.sum(
        dim=-1
    )

    # --------------------------------------------------------
    # P(K)
    # --------------------------------------------------------

    k_logits = masked_k_logits(
        params.k_logits,
        available_count,
    )

    k_logp = F.log_softmax(
        k_logits,
        dim=-1,
    ).gather(
        -1,
        action.K.unsqueeze(-1),
    ).squeeze(-1)

    # --------------------------------------------------------
    # P(selected items | K)
    # --------------------------------------------------------

    items_logp = item_order_log_prob(
        params.item_logits,
        action.item_order,
        available,
        action.K,
    )

    # --------------------------------------------------------
    # P(amounts | selected items)
    # --------------------------------------------------------

    selected = (
        action.quantities > 0
    )

    amount_logp = positive_gpd_log_prob(
        q=action.quantities.clamp_min(1),
        location=params.item_logits,
        scale=params.amount_scale,
        shape=params.amount_shape,
        maximum=maximum.clamp_min(1),
    )

    amount_logp = torch.where(
        selected,
        amount_logp,
        torch.zeros_like(amount_logp),
    ).sum(dim=-1)

    return (
        k_logp
        + items_logp
        + amount_logp
    )


# ============================================================
# Complete BUY + SELL barter
# ============================================================

def sample_barter(
    params: BarterParams,
    buy_maximum: Tensor,
    sell_maximum: Tensor,
) -> BarterAction:

    return BarterAction(
        buy=sample_barter_side(
            params.buy,
            buy_maximum,
        ),
        sell=sample_barter_side(
            params.sell,
            sell_maximum,
        ),
    )


def barter_log_prob(
    params: BarterParams,
    action: BarterAction,
    buy_maximum: Tensor,
    sell_maximum: Tensor,
) -> Tensor:

    buy_logp = barter_side_log_prob(
        params.buy,
        action.buy,
        buy_maximum,
    )

    sell_logp = barter_side_log_prob(
        params.sell,
        action.sell,
        sell_maximum,
    )

    return (
        buy_logp
        + sell_logp
    )


# ============================================================
# Merchant / city categorical distributions
# ============================================================

def merchant_logits(
    params: ActionParams,
    city_merchants: Tensor,
) -> Tensor:
    """
    city_merchants:
        [..., NUM_MERCHANTS, D_MODEL]
    """

    return torch.einsum(
        "...nd,...d->...n",
        city_merchants,
        params.merchant_query,
    )


def city_logits(
    params: ActionParams,
    neighbor_cities: Tensor,
) -> Tensor:
    """
    neighbor_cities:
        [..., NUM_CITIES, D_MODEL]
    """

    return torch.einsum(
        "...nd,...d->...n",
        neighbor_cities,
        params.city_query,
    )


def sample_merchant(
    params: ActionParams,
    city_merchants: Tensor,
) -> Tensor:

    return torch.distributions.Categorical(
        logits=merchant_logits(
            params,
            city_merchants,
        )
    ).sample()


def merchant_log_prob(
    params: ActionParams,
    city_merchants: Tensor,
    merchant: Tensor,
) -> Tensor:

    return torch.distributions.Categorical(
        logits=merchant_logits(
            params,
            city_merchants,
        )
    ).log_prob(merchant)


def sample_city(
    params: ActionParams,
    neighbor_cities: Tensor,
) -> Tensor:

    return torch.distributions.Categorical(
        logits=city_logits(
            params,
            neighbor_cities,
        )
    ).sample()


def city_log_prob(
    params: ActionParams,
    neighbor_cities: Tensor,
    city: Tensor,
) -> Tensor:

    return torch.distributions.Categorical(
        logits=city_logits(
            params,
            neighbor_cities,
        )
    ).log_prob(city)

# Test
this is LLM made I did not fully test in depth the statistics yet, its mostly here as a temporary sanity check

In [7]:
# test_action_statistics.py

import itertools
from collections import Counter

import torch

# from action_layer import (
#     BarterSideParams,
#     BarterSideAction,
#     TradeCountParams,

#     sample_positive_gpd,
#     positive_gpd_log_prob,

#     sample_trade_count,
#     trade_count_log_prob,

#     sample_item_order,
#     item_order_log_prob,

#     sample_barter_side,
#     barter_side_log_prob,
# )


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

N = 1_000_000


# ============================================================
# Small helpers
# ============================================================

def check_distribution(
    observed,
    expected,
    *,
    atol=0.003,
    name="distribution",
):
    """
    observed/expected are 1D probability tensors.
    """

    observed = observed.cpu()
    expected = expected.cpu()

    print(f"\n{name}")
    print("value      expected      observed      error")

    for i, (e, o) in enumerate(zip(expected, observed)):
        print(
            f"{i:5d}      "
            f"{e.item():.6f}      "
            f"{o.item():.6f}      "
            f"{abs(e.item() - o.item()):.6f}"
        )

    max_error = (expected - observed).abs().max().item()

    print("max error:", max_error)
    print("expected sum:", expected.sum().item())
    print("observed sum:", observed.sum().item())

    assert abs(expected.sum().item() - 1.0) < 1e-5
    assert abs(observed.sum().item() - 1.0) < 1e-5
    assert max_error < atol


# ============================================================
# Test 1:
#
# sample_positive_gpd()
#        vs
# positive_gpd_log_prob()
#
# These are the two things we ACTUALLY care agree.
# ============================================================

def test_positive_gpd_sample_vs_log_prob():
    cases = [
        # location, scale, shape, maximum
        (0.0,  1.0, 0.10, 10),
        (1.7,  3.0, 0.20, 15),
        (5.0,  2.0, 0.35, 20),
        (-2.0, 7.0, 0.45, 20),
        (15.0, 4.0, 0.25, 25),
    ]

    for location, scale, shape, maximum in cases:

        # ------------------------------
        # Monte Carlo side
        # ------------------------------

        samples = sample_positive_gpd(
            location=torch.full(
                (N,),
                location,
                device=DEVICE,
            ),
            scale=torch.full(
                (N,),
                scale,
                device=DEVICE,
            ),
            shape=torch.full(
                (N,),
                shape,
                device=DEVICE,
            ),
            maximum=torch.full(
                (N,),
                maximum,
                dtype=torch.long,
                device=DEVICE,
            ),
        )

        counts = torch.bincount(
            samples.cpu(),
            minlength=maximum + 1,
        ).float()

        observed = counts[1:] / N

        # ------------------------------
        # log_prob side
        # ------------------------------

        q = torch.arange(
            1,
            maximum + 1,
            device=DEVICE,
        )

        expected = positive_gpd_log_prob(
            q=q,
            location=torch.tensor(
                location,
                device=DEVICE,
            ),
            scale=torch.tensor(
                scale,
                device=DEVICE,
            ),
            shape=torch.tensor(
                shape,
                device=DEVICE,
            ),
            maximum=torch.tensor(
                maximum,
                device=DEVICE,
            ),
        ).exp()

        check_distribution(
            observed,
            expected,
            name=(
                f"positive GPD "
                f"loc={location}, "
                f"scale={scale}, "
                f"shape={shape}, "
                f"max={maximum}"
            ),
        )


# ============================================================
# Test 2:
#
# Random fuzzing of exactly the same sample-vs-log_prob
# relationship.
# ============================================================

def test_random_positive_gpd():
    torch.manual_seed(1234)

    for test_id in range(20):

        location = (
            torch.randn(()).item() * 8
        )

        scale = (
            torch.rand(()).item() * 10
            + 0.1
        )

        shape = (
            torch.rand(()).item() * 0.48
            + 0.005
        )

        maximum = int(
            torch.randint(
                2,
                40,
                (),
            ).item()
        )

        n = 400_000

        samples = sample_positive_gpd(
            torch.full(
                (n,),
                location,
                device=DEVICE,
            ),
            torch.full(
                (n,),
                scale,
                device=DEVICE,
            ),
            torch.full(
                (n,),
                shape,
                device=DEVICE,
            ),
            torch.full(
                (n,),
                maximum,
                dtype=torch.long,
                device=DEVICE,
            ),
        )

        observed = (
            torch.bincount(
                samples.cpu(),
                minlength=maximum + 1,
            ).float()[1:]
            / n
        ).to(DEVICE)

        q = torch.arange(
            1,
            maximum + 1,
            device=DEVICE,
        )

        expected = positive_gpd_log_prob(
            q,
            torch.tensor(location, device=DEVICE),
            torch.tensor(scale, device=DEVICE),
            torch.tensor(shape, device=DEVICE),
            torch.tensor(maximum, device=DEVICE),
        ).exp()

        error = (
            observed - expected
        ).abs().max().item()

        print(
            f"random {test_id:2d}: "
            f"loc={location:7.3f} "
            f"scale={scale:6.3f} "
            f"shape={shape:.3f} "
            f"max={maximum:2d} "
            f"error={error:.5f}"
        )

        assert abs(expected.sum().item() - 1) < 1e-5
        assert error < 0.006


# ============================================================
# Test 3:
#
# Existing-trade count.
#
# This includes BOTH:
#
#     P(count = 0)
#
# and
#
#     positive GPD count conditional on execution.
# ============================================================

def test_trade_count_sample_vs_log_prob():
    cases = [
        (-3.0, 0.0, 1.0, 0.10, 10),
        (-1.0, 2.0, 3.0, 0.20, 15),
        ( 0.0, 5.0, 2.0, 0.30, 20),
        ( 2.0, 1.0, 8.0, 0.40, 25),
        ( 5.0, 8.0, 3.0, 0.45, 30),
    ]

    for (
        execute_logit,
        location,
        scale,
        shape,
        maximum,
    ) in cases:

        # ------------------------------
        # Monte Carlo
        # ------------------------------

        params = TradeCountParams(
            execute_logit=torch.full(
                (N,),
                execute_logit,
                device=DEVICE,
            ),
            location=torch.full(
                (N,),
                location,
                device=DEVICE,
            ),
            scale=torch.full(
                (N,),
                scale,
                device=DEVICE,
            ),
            shape=torch.full(
                (N,),
                shape,
                device=DEVICE,
            ),
        )

        samples = sample_trade_count(
            params,
            torch.full(
                (N,),
                maximum,
                dtype=torch.long,
                device=DEVICE,
            ),
        )

        observed = (
            torch.bincount(
                samples.cpu(),
                minlength=maximum + 1,
            ).float()
            / N
        )

        # ------------------------------
        # log_prob
        # ------------------------------

        scalar_params = TradeCountParams(
            execute_logit=torch.tensor(
                execute_logit,
                device=DEVICE,
            ),
            location=torch.tensor(
                location,
                device=DEVICE,
            ),
            scale=torch.tensor(
                scale,
                device=DEVICE,
            ),
            shape=torch.tensor(
                shape,
                device=DEVICE,
            ),
        )

        values = torch.arange(
            maximum + 1,
            device=DEVICE,
        )

        expected = trade_count_log_prob(
            scalar_params,
            values,
            torch.tensor(
                maximum,
                device=DEVICE,
            ),
        ).exp()

        check_distribution(
            observed,
            expected,
            name=(
                f"trade count "
                f"logit={execute_logit}, "
                f"loc={location}, "
                f"scale={scale}, "
                f"shape={shape}, "
                f"max={maximum}"
            ),
        )


# ============================================================
# Test 4:
#
# Item selection.
#
# Don't manually reproduce Plackett-Luce.
#
# Instead:
#
#   1. sample a LOT of complete orders
#   2. histogram each possible order
#   3. ask item_order_log_prob() for each order
#
# and compare.
# ============================================================

def test_item_order_sample_vs_log_prob():
    logits = torch.tensor(
        [0.2, 1.1, -0.5],
        device=DEVICE,
    )

    available = torch.tensor(
        [True, True, True],
        device=DEVICE,
    )

    K = 2

    # ------------------------------
    # Monte Carlo
    # ------------------------------

    sampled_orders = sample_item_order(
        logits.expand(N, -1),
        available.expand(N, -1),
    )

    # Only the first K selections matter.
    sampled_orders = sampled_orders[:, :K].cpu()

    counts = Counter(
        tuple(row.tolist())
        for row in sampled_orders
    )

    # ------------------------------
    # Enumerate every possible
    # ordered selection.
    # ------------------------------

    possible = list(
        itertools.permutations(
            range(3),
            K,
        )
    )

    expected_probs = []
    observed_probs = []

    print("\nitem order probabilities")

    for prefix in possible:

        # item_order_log_prob expects a complete ordering,
        # even though only first K entries contribute.
        remainder = [
            i
            for i in range(3)
            if i not in prefix
        ]

        full_order = torch.tensor(
            list(prefix) + remainder,
            device=DEVICE,
        )

        logp = item_order_log_prob(
            item_logits=logits,
            item_order=full_order,
            available=available,
            K=torch.tensor(
                K,
                device=DEVICE,
            ),
        )

        expected = logp.exp().item()

        observed = (
            counts[prefix] / N
        )

        expected_probs.append(expected)
        observed_probs.append(observed)

        print(
            prefix,
            "expected:",
            f"{expected:.6f}",
            "observed:",
            f"{observed:.6f}",
        )

    expected_probs = torch.tensor(
        expected_probs
    )

    observed_probs = torch.tensor(
        observed_probs
    )

    assert abs(
        expected_probs.sum().item() - 1
    ) < 1e-5

    assert (
        expected_probs
        - observed_probs
    ).abs().max() < 0.004


# ============================================================
# Test 5:
#
# WHOLE barter side.
#
# This is the strongest test.
#
# Use:
#
#   NUM_ITEMS = 2
#   maximum = [2, 2]
#
# so the COMPLETE action space is tiny enough that we can
# Monte Carlo it.
#
# We compare:
#
#     empirical frequency of complete actions
#
# against
#
#     exp(barter_side_log_prob(action))
#
# ============================================================

def test_complete_barter_side():
    n = 2_000_000

    params = BarterSideParams(
        k_logits=torch.tensor(
            [0.3, 1.0, 0.5],
            device=DEVICE,
        ),

        item_logits=torch.tensor(
            [0.7, -0.2],
            device=DEVICE,
        ),

        amount_scale=torch.tensor(
            [1.3, 2.0],
            device=DEVICE,
        ),

        amount_shape=torch.tensor(
            [0.15, 0.35],
            device=DEVICE,
        ),
    )

    maximum = torch.tensor(
        [2, 2],
        device=DEVICE,
    )

    # Expand params for vectorized MC sampling.
    batch_params = BarterSideParams(
        k_logits=params.k_logits.expand(n, -1),
        item_logits=params.item_logits.expand(n, -1),
        amount_scale=params.amount_scale.expand(n, -1),
        amount_shape=params.amount_shape.expand(n, -1),
    )

    batch_maximum = maximum.expand(
        n,
        -1,
    )

    actions = sample_barter_side(
        batch_params,
        batch_maximum,
    )

    # --------------------------------------------------------
    # Count complete sampling paths.
    #
    # IMPORTANT:
    #
    # The policy action includes the ordered selected path,
    # because that's what barter_side_log_prob() scores.
    # --------------------------------------------------------

    observed_counts = Counter()

    quantities_cpu = actions.quantities.cpu()
    K_cpu = actions.K.cpu()
    order_cpu = actions.item_order.cpu()

    for i in range(n):
        k = int(K_cpu[i])

        key = (
            tuple(
                quantities_cpu[i].tolist()
            ),
            k,
            tuple(
                order_cpu[i, :k].tolist()
            ),
        )

        observed_counts[key] += 1

    # --------------------------------------------------------
    # Re-score every action that actually appeared.
    # --------------------------------------------------------

    max_error = 0.0
    expected_total = 0.0

    print(
        "\ncomplete barter-side actions "
        "(showing reasonably common ones)"
    )

    for key, count in observed_counts.items():
        quantities, k, selected_prefix = key

        # Complete item order.
        remainder = [
            i
            for i in range(2)
            if i not in selected_prefix
        ]

        full_order = (
            list(selected_prefix)
            + remainder
        )

        action = BarterSideAction(
            quantities=torch.tensor(
                quantities,
                device=DEVICE,
            ),
            K=torch.tensor(
                k,
                device=DEVICE,
            ),
            item_order=torch.tensor(
                full_order,
                device=DEVICE,
            ),
        )

        expected = torch.exp(
            barter_side_log_prob(
                params,
                action,
                maximum,
            )
        ).item()

        observed = count / n

        expected_total += expected

        error = abs(
            expected - observed
        )

        max_error = max(
            max_error,
            error,
        )

        if expected > 0.005:
            print(
                key,
                f"expected={expected:.6f}",
                f"observed={observed:.6f}",
                f"error={error:.6f}",
            )

    print("max error:", max_error)
    print("expected total:", expected_total)

    assert max_error < 0.003

    # Since with these tiny limits every action should have
    # shown up in 2M samples, this should be essentially 1.
    assert abs(
        expected_total - 1.0
    ) < 1e-5


# ============================================================
# Main
# ============================================================

def main():
    torch.manual_seed(12345)

    print("device:", DEVICE)

    test_positive_gpd_sample_vs_log_prob()
    test_random_positive_gpd()
    test_trade_count_sample_vs_log_prob()
    test_item_order_sample_vs_log_prob()
    test_complete_barter_side()

    print("\nALL STATISTICAL TESTS PASSED")


if __name__ == "__main__":
    main()

device: cuda

positive GPD loc=0.0, scale=1.0, shape=0.1, max=10
value      expected      observed      error
    0      0.838494      0.837961      0.000533
    1      0.088967      0.089207      0.000240
    2      0.037967      0.037995      0.000028
    3      0.017230      0.017202      0.000028
    4      0.008247      0.008415      0.000168
    5      0.004135      0.004250      0.000115
    6      0.002160      0.002103      0.000057
    7      0.001170      0.001249      0.000079
    8      0.000654      0.000617      0.000037
    9      0.000977      0.001001      0.000024
max error: 0.0005334615707397461
expected sum: 1.0
observed sum: 1.0

positive GPD loc=1.7, scale=3.0, shape=0.2, max=15
value      expected      observed      error
    0      0.094269      0.094128      0.000141
    1      0.245770      0.245425      0.000345
    2      0.169927      0.169921      0.000006
    3      0.120034      0.120144      0.000110
    4      0.086423      0.086185      0.000238
    

# Rest

In [10]:
class Model(nn.Module):
    def __init__(self):
        super().__init__()

        self.merchants = AproxMemory(D_MODEL,D_MODEL,D_MODEL,MAX_OTHERS)
        self.cities = AproxMemory(D_MODEL,D_MODEL,D_MODEL,MAX_SPOTS)
        self.act = ActionLayer(D_MODEL)
    
    def forward(self,x):
        pass

Model()

Model(
  (merchants): AproxMemory(
    (x_proj): Linear(in_features=64, out_features=192, bias=True)
    (h_proj): Linear(in_features=64, out_features=192, bias=False)
    (out_proj): Linear(in_features=64, out_features=64, bias=True)
  )
  (cities): AproxMemory(
    (x_proj): Linear(in_features=64, out_features=192, bias=True)
    (h_proj): Linear(in_features=64, out_features=192, bias=False)
    (out_proj): Linear(in_features=64, out_features=64, bias=True)
  )
  (act): ActionLayer(
    (f): Linear(in_features=64, out_features=153, bias=True)
  )
)